# Modeling (V38) — V28 + New Timing Features

**Changes from V28:** 4 new engineered features added to the feature set:
- `log_dt = log1p(dt_first_last_0_5h)` — log transform of observation window (r=-0.489 with log(T_close), vs raw r=-0.475)
- `perimeter_rate = (num_perimeters-1) / (dt_first_last+0.1)` — observation density (r=-0.314, genuinely new signal)
- `num_x_low_res = num_perimeters × low_temporal_resolution` — interaction (r=+0.478)
- `low_res_x_align = low_temporal_resolution × alignment_abs` — interaction (r=+0.20)

**Architecture:** Same as V28 — backbone (RSF+GBM+Coxnet), 5-seed close-fire Coxnet (alpha=0.20, l1=0.5), beta blend, Platt/isotonic calibration.

**Key results:** OOF=0.9737 (+0.0003 vs V28)  C=0.9451  WBS=0.0141  beta=0.60  0 violations

**Hard rules (unchanged from V28):**
- alpha=0.20, l1_ratio=0.5 LOCKED for close-fire Coxnet
- Distance threshold=5000m LOCKED
- MIN_COXNET_W=0.60 LOCKED

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

In [2]:
SEED           = 42
N_SPLITS       = 10
N_SPLITS_CLOSE = 7
DIST_THRESHOLD = 5_000
HORIZONS       = [12, 24, 48, 72]
MODEL_NAMES    = ['rsf', 'gbm', 'coxnet']
MIN_COXNET_W   = 0.60
COXNET_ALPHA   = 0.05
CLOSE_ALPHA    = 0.20
CLOSE_L1       = 0.5
CLOSE_SEEDS    = [42, 100, 200, 300, 400]
BETA_GRID      = [round(i * 0.05, 2) for i in range(13)]

BASE_FEATURES = [
    'num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h',
    'area_first_ha', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth',
    'radial_growth_rate_m_per_h', 'area_growth_rel_0_5h', 'log_area_ratio_0_5h',
    'centroid_speed_m_per_h', 'spread_bearing_sin', 'spread_bearing_cos',
    'centroid_displacement_m', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h',
    'projected_advance_m', 'dist_fit_r2_0_5h', 'dist_accel_m_per_h2',
    'dist_slope_ci_0_5h', 'alignment_cos', 'alignment_abs',
]
ENGINEERED = [
    'log_dist_min', 'log_dist_sq', 'dist_close_5k', 'dist_close_30k',
    'tti_h', 'hazard_proxy', 'is_dynamic', 'is_closing', 'dynamic_x_dist',
    'dist_x_low_res', 'growth_momentum',
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'align_x_month_sin', 'bearing_x_month_cos', 'growth_x_dow_sin',
]
# V38 additions
NEW_FEATURES = ['log_dt', 'perimeter_rate', 'num_x_low_res', 'low_res_x_align']

In [3]:
os.chdir('/Users/marcelloborromeo/Downloads/BTT WiDS Datathon')

def engineer(df: pd.DataFrame) -> pd.DataFrame:
    """V38: V8-exact 42 features + 4 new timing features."""
    df = df.copy()
    eps = 1.0
    raw_dist = df['dist_min_ci_0_5h']

    df['log_dist_min']   = np.log1p(raw_dist)
    df['log_dist_sq']    = df['log_dist_min'] ** 2
    df['dist_close_5k']  = (raw_dist < 5_000).astype(float)
    df['dist_close_30k'] = (raw_dist < 30_000).astype(float)

    closing = df['closing_speed_m_per_h'].clip(lower=0)
    df['tti_h']          = (raw_dist / (closing + eps)).clip(upper=500)
    df['hazard_proxy']   = df['alignment_abs'] * closing / (raw_dist + eps)
    df['is_dynamic']     = (df['area_growth_rate_ha_per_h'] > 0).astype(float)
    df['is_closing']     = (df['closing_speed_m_per_h'] > 0).astype(float)
    df['dynamic_x_dist'] = df['is_dynamic'] * df['log_dist_min']
    df['dist_x_low_res'] = df['log_dist_min'] * df['low_temporal_resolution_0_5h']
    df['growth_momentum'] = df['log1p_area_first'] * df['radial_growth_rate_m_per_h']

    df['month_sin'] = np.sin(2 * np.pi * df['event_start_month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['event_start_month'] / 12)
    df['hour_sin']  = np.sin(2 * np.pi * df['event_start_hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['event_start_hour'] / 24)
    df['dow_sin']   = np.sin(2 * np.pi * df['event_start_dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['event_start_dayofweek'] / 7)

    df['align_x_month_sin']   = df['alignment_abs'] * df['month_sin']
    df['bearing_x_month_cos'] = df['spread_bearing_cos'] * df['month_cos']
    df['growth_x_dow_sin']    = df['log1p_growth'] * df['dow_sin']

    # V38 new features
    df['log_dt']          = np.log1p(df['dt_first_last_0_5h'])
    df['perimeter_rate']  = (df['num_perimeters_0_5h'] - 1) / (df['dt_first_last_0_5h'] + 0.1)
    df['num_x_low_res']   = df['num_perimeters_0_5h'] * df['low_temporal_resolution_0_5h']
    df['low_res_x_align'] = df['low_temporal_resolution_0_5h'] * df['alignment_abs']
    return df

train    = pd.read_csv('train.csv')
test     = pd.read_csv('test.csv')
train_fe = engineer(train)
test_fe  = engineer(test)

FEAT_COLS = [f for f in BASE_FEATURES + ENGINEERED + NEW_FEATURES if f in train_fe.columns]
print(f'Feature count: {len(FEAT_COLS)} (42 original + {len(NEW_FEATURES)} new)')

y_event = train['event'].values.astype(bool)
y_time  = train['time_to_hit_hours'].values.astype(float)

close_train = train_fe['dist_min_ci_0_5h'].values < DIST_THRESHOLD
close_test  = test_fe['dist_min_ci_0_5h'].values  < DIST_THRESHOLD
print(f'Train close fires: {close_train.sum()}  |  Test close fires: {close_test.sum()}')

pre = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
X_train = pre.fit_transform(train_fe[FEAT_COLS].values)
X_test  = pre.transform(test_fe[FEAT_COLS].values)
y_surv  = Surv.from_arrays(y_event, y_time)

Feature count: 46 (42 original + 4 new)
Train close fires: 69  |  Test close fires: 28


In [4]:
def surv_fn_to_probs(surv_fns) -> np.ndarray:
    out = np.zeros((len(surv_fns), len(HORIZONS)))
    for i, sf in enumerate(surv_fns):
        t_min, t_max = sf.domain
        for j, t in enumerate(HORIZONS):
            out[i, j] = 1.0 - float(sf(float(np.clip(t, t_min, t_max))))
    return out

def brier_at(p_h, y_evt, y_t, h):
    hit  = (y_evt == 1) & (y_t <= h)
    excl = (y_evt == 0) & (y_t <  h)
    keep = ~excl
    if keep.sum() == 0:
        return np.nan
    return float(np.mean((p_h[keep] - hit[keep].astype(float)) ** 2))

def hybrid_score(probs, y_evt, y_t, risk=None):
    wbs  = (0.3 * brier_at(probs[:, 1], y_evt, y_t, 24) +
            0.4 * brier_at(probs[:, 2], y_evt, y_t, 48) +
            0.3 * brier_at(probs[:, 3], y_evt, y_t, 72))
    risk = probs[:, 2] if risk is None else risk
    try:
        c, *_ = concordance_index_censored(y_evt.astype(bool), y_t, risk)
    except Exception:
        c = 0.5
    return 0.3 * c + 0.7 * (1 - wbs), c, wbs

def platt_fit(raw_h, y_evt, y_t, h):
    hit  = (y_evt == 1) & (y_t <= h)
    excl = (y_evt == 0) & (y_t <  h)
    keep = ~excl
    lr = LogisticRegression(C=1e6, solver='lbfgs', max_iter=2000)
    lr.fit(raw_h[keep].reshape(-1, 1), hit[keep].astype(int))
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])

def iso_fit(raw_h, y_evt, y_t, h):
    hit  = (y_evt == 1) & (y_t <= h)
    excl = (y_evt == 0) & (y_t <  h)
    keep = ~excl
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(raw_h[keep], hit[keep].astype(float))
    return iso

def apply_calibration(raw, platts, iso48, iso72):
    a12, b12 = platts[12]
    a24, b24 = platts[24]
    out = np.column_stack([
        expit(a12 * raw[:, 0] + b12),
        expit(a24 * raw[:, 1] + b24),
        iso48.predict(raw[:, 2]),
        iso72.predict(raw[:, 3]),
    ])
    return np.clip(np.maximum.accumulate(out, axis=1), 0.0, 1.0)

def check_monotonicity(arr):
    return int((np.diff(arr, axis=1) < 0).any())

In [5]:
def build_rsf(n_estimators=200):
    return RandomSurvivalForest(
        n_estimators=n_estimators, max_depth=4, min_samples_leaf=10,
        max_features='sqrt', n_jobs=-1, random_state=SEED,
    )

def build_gbm():
    return GradientBoostingSurvivalAnalysis(
        n_estimators=150, learning_rate=0.04, max_depth=2,
        subsample=0.8, min_samples_leaf=15, random_state=SEED,
    )

def build_coxnet_backbone():
    return CoxnetSurvivalAnalysis(
        alphas=[COXNET_ALPHA], l1_ratio=0.5,
        fit_baseline_model=True, max_iter=10_000,
    )

def build_coxnet_close():
    """Close-fire timing Coxnet — alpha=0.20, l1=0.5 LOCKED."""
    return CoxnetSurvivalAnalysis(
        alphas=[CLOSE_ALPHA], l1_ratio=CLOSE_L1,
        fit_baseline_model=True, max_iter=10_000,
    )

In [6]:
print('Stage 1: Backbone survival OOF (10-fold StratifiedKFold)...')
np.random.seed(SEED)

n   = len(X_train)
oof = {name: np.zeros((n, len(HORIZONS))) for name in MODEL_NAMES}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fold, (tr_i, va_i) in enumerate(skf.split(X_train, y_event.astype(int))):
    X_tr, X_va = X_train[tr_i], X_train[va_i]
    y_tr = y_surv[tr_i]

    m = build_rsf()
    m.fit(X_tr, y_tr)
    oof['rsf'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))

    m = build_gbm()
    m.fit(X_tr, y_tr)
    oof['gbm'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))

    try:
        m = build_coxnet_backbone()
        m.fit(X_tr, y_tr)
        oof['coxnet'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))
    except Exception:
        oof['coxnet'][va_i] = oof['rsf'][va_i].copy()

print('  Backbone OOF complete.')

Stage 1: Backbone survival OOF (10-fold StratifiedKFold)...


  Backbone OOF complete.


In [7]:
print('Nelder-Mead 3-way blend (MIN_COXNET_W=0.60)...')

coxnet_i = MODEL_NAMES.index('coxnet')
rng      = np.random.RandomState(SEED + 1)
starts   = [np.ones(3) / 3] + [rng.dirichlet(np.ones(3)) for _ in range(9)]
v8like   = np.zeros(3); v8like[coxnet_i] = 0.79; v8like[1] = 0.21
starts.append(v8like)

def blend_obj(w_raw):
    w = np.abs(w_raw); s = w.sum()
    if s < 1e-9: return 1.0
    w /= s
    b    = sum(w[i] * oof[MODEL_NAMES[i]] for i in range(3))
    wbs  = (0.3 * brier_at(b[:, 1], y_event, y_time, 24) +
            0.4 * brier_at(b[:, 2], y_event, y_time, 48) +
            0.3 * brier_at(b[:, 3], y_event, y_time, 72))
    try:
        c, *_ = concordance_index_censored(y_event.astype(bool), y_time, b[:, 2])
    except Exception:
        c = 0.5
    return -(0.3 * c + 0.7 * (1 - wbs))

best_obj = np.inf; best_w = None
for w0 in starts:
    res = minimize(blend_obj, w0, method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-7, 'fatol': 1e-8})
    w = np.abs(res.x); w /= w.sum()
    if res.fun < best_obj and w[coxnet_i] >= MIN_COXNET_W:
        best_obj = res.fun; best_w = w.copy()

if best_w is None:
    best_w = np.array([0.044, 0.178, 0.778])
    print('  WARNING: constraint not met, using V8 fallback weights')

weights       = dict(zip(MODEL_NAMES, best_w))
raw_blend_oof = sum(weights[nm] * oof[nm] for nm in MODEL_NAMES)

print(f'  Weights: coxnet={weights["coxnet"]:.3f}  gbm={weights["gbm"]:.3f}  rsf={weights["rsf"]:.3f}')
h_back, c_back, wbs_back = hybrid_score(raw_blend_oof, y_event, y_time)
print(f'  Backbone OOF: hybrid={h_back:.4f}  C={c_back:.4f}  WBS={wbs_back:.4f}')

Nelder-Mead 3-way blend (MIN_COXNET_W=0.60)...


  Weights: coxnet=0.720  gbm=0.205  rsf=0.075
  Backbone OOF: hybrid=0.9707  C=0.9430  WBS=0.0174


In [8]:
print('Stage 2: Close-fire Coxnet OOF (5-seed average, 7-fold KFold)...')

X_close      = X_train[close_train]
y_surv_close = y_surv[close_train]
n_close      = close_train.sum()

seed_oofs = []
for s in CLOSE_SEEDS:
    kf       = KFold(n_splits=N_SPLITS_CLOSE, shuffle=True, random_state=s)
    seed_oof = np.zeros((n_close, len(HORIZONS)))
    for tr, va in kf.split(X_close):
        try:
            m = build_coxnet_close()
            m.fit(X_close[tr], y_surv_close[tr])
            seed_oof[va] = surv_fn_to_probs(m.predict_survival_function(X_close[va]))
        except Exception:
            seed_oof[va] = raw_blend_oof[close_train][va]
    seed_oofs.append(seed_oof)
    print(f'  Seed {s} done.')

close_cox_oof = np.mean(seed_oofs, axis=0)
print(f'  Close-fire Coxnet OOF complete ({n_close} fires, {len(CLOSE_SEEDS)} seeds).')

Stage 2: Close-fire Coxnet OOF (5-seed average, 7-fold KFold)...
  Seed 42 done.
  Seed 100 done.
  Seed 200 done.
  Seed 300 done.
  Seed 400 done.
  Close-fire Coxnet OOF complete (69 fires, 5 seeds).


In [9]:
print('Stage 3: Beta blend search + Platt/isotonic calibration...')

best_h    = -np.inf
best_beta = 0
best_cals = None
best_c    = 0
best_wbs  = 0

for beta in BETA_GRID:
    mixed = raw_blend_oof.copy()
    mixed[close_train] = (1 - beta) * raw_blend_oof[close_train] + beta * close_cox_oof

    platts = {
        12: platt_fit(mixed[:, 0], y_event, y_time, 12),
        24: platt_fit(mixed[:, 1], y_event, y_time, 24),
    }
    iso48 = iso_fit(mixed[:, 2], y_event, y_time, 48)
    iso72 = iso_fit(mixed[:, 3], y_event, y_time, 72)

    cal = apply_calibration(mixed, platts, iso48, iso72)
    h, c, wbs = hybrid_score(cal, y_event, y_time, risk=mixed[:, 2])

    if h > best_h:
        best_h    = h
        best_beta = beta
        best_c    = c
        best_wbs  = wbs
        best_cals = {'platts': platts, 'iso48': iso48, 'iso72': iso72}

print(f'  Best beta: {best_beta}')
print(f'  OOF hybrid: {best_h:.4f}  C: {best_c:.4f}  WBS: {best_wbs:.4f}')
print(f'  V28 reference: OOF=0.9734  LB=0.97458')

Stage 3: Beta blend search + Platt/isotonic calibration...
  Best beta: 0.6
  OOF hybrid: 0.9737  C: 0.9451  WBS: 0.0141
  V28 reference: OOF=0.9734  LB=0.97458


In [10]:
versions = {
    'V8  (LB=0.97090)': {'c': 0.9403, 'wbs': 0.0148, 'h': 0.9717},
    'V21 (LB=0.97438)': {'c': 0.9448, 'wbs': 0.0143, 'h': 0.9729},
    'V28 (LB=0.97458)': {'c': 0.9448, 'wbs': 0.0143, 'h': 0.9734},
    'V38 (this run)':   {'c': best_c,  'wbs': best_wbs, 'h': best_h},
}
print(f'{"Version":<22} {"Hybrid":>8} {"C-index":>9} {"WBS":>8}')
print('-' * 52)
for vname, m in versions.items():
    print(f'{vname:<22} {m["h"]:>8.4f} {m["c"]:>9.4f} {m["wbs"]:>8.4f}')
print(f'\nV38 delta vs V28: {best_h - 0.9734:+.4f}')

Version                  Hybrid   C-index      WBS
----------------------------------------------------
V8  (LB=0.97090)         0.9717    0.9403   0.0148
V21 (LB=0.97438)         0.9729    0.9448   0.0143
V28 (LB=0.97458)         0.9734    0.9448   0.0143
V38 (this run)           0.9737    0.9451   0.0141

V38 delta vs V28: +0.0003


In [11]:
print('Training final models on all 221 examples...')

m_rsf = build_rsf(n_estimators=400)
m_rsf.fit(X_train, y_surv)

m_gbm = build_gbm()
m_gbm.fit(X_train, y_surv)

m_cox_backbone = build_coxnet_backbone()
m_cox_backbone.fit(X_train, y_surv)

final_models = {'rsf': m_rsf, 'gbm': m_gbm, 'coxnet': m_cox_backbone}
print('  Backbone final models trained.')

m_cox_close = build_coxnet_close()
m_cox_close.fit(X_close, y_surv_close)
print('  Close-fire Coxnet final model trained.')

Training final models on all 221 examples...


  Backbone final models trained.
  Close-fire Coxnet final model trained.


In [12]:
print('Generating test predictions...')

test_raw = sum(
    weights[nm] * surv_fn_to_probs(final_models[nm].predict_survival_function(X_test))
    for nm in MODEL_NAMES
)

X_test_close      = X_test[close_test]
close_test_probs  = surv_fn_to_probs(m_cox_close.predict_survival_function(X_test_close))

test_mixed = test_raw.copy()
test_mixed[close_test] = (
    (1 - best_beta) * test_raw[close_test] + best_beta * close_test_probs
)

test_cal = apply_calibration(test_mixed, best_cals['platts'], best_cals['iso48'], best_cals['iso72'])
print('  Test predictions generated.')

Generating test predictions...
  Test predictions generated.


In [13]:
print('=== Submission Monotonicity Check ===')
v_12_24 = int((test_cal[:, 0] > test_cal[:, 1]).sum())
v_24_48 = int((test_cal[:, 1] > test_cal[:, 2]).sum())
v_48_72 = int((test_cal[:, 2] > test_cal[:, 3]).sum())
total_v = v_12_24 + v_24_48 + v_48_72
print(f'  12->24h violations: {v_12_24}')
print(f'  24->48h violations: {v_24_48}')
print(f'  48->72h violations: {v_48_72}')
print(f'  Total: {total_v}')
assert total_v == 0, f'Monotonicity violated! {total_v} violations.'

sub = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h':  test_cal[:, 0],
    'prob_24h':  test_cal[:, 1],
    'prob_48h':  test_cal[:, 2],
    'prob_72h':  test_cal[:, 3],
})
sub_path = 'Artifacts and Submissions/submission (V38_new_features).csv'
sub.to_csv(sub_path, index=False)
print(f'  Saved: {sub_path}')
print(f'  Shape: {sub.shape}')
print(sub.head())

=== Submission Monotonicity Check ===
  12->24h violations: 0
  24->48h violations: 0
  48->72h violations: 0
  Total: 0
  Saved: Artifacts and Submissions/submission (V38_new_features).csv
  Shape: (95, 5)
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.006360  0.006360  0.006360       1.0
1  13353600  0.456720  0.874990  1.000000       1.0
2  13942327  0.006621  0.006621  0.006621       1.0
3  16112781  0.518614  0.897732  1.000000       1.0
4  17132808  0.011214  0.011214  0.011214       1.0


In [14]:
artifacts = {
    'version':        'V38',
    'feat_cols':      FEAT_COLS,
    'new_features':   NEW_FEATURES,
    'preprocessor':   pre,
    'weights':        weights,
    'best_beta':      best_beta,
    'best_cals':      best_cals,
    'final_models':   final_models,
    'close_model':    m_cox_close,
    'close_train':    close_train,
    'close_test':     close_test,
    'close_seeds':    CLOSE_SEEDS,
    'oof_hybrid':     best_h,
    'oof_c':          best_c,
    'oof_wbs':        best_wbs,
}
art_path = 'Artifacts and Submissions/model_artifacts (V38).pkl'
with open(art_path, 'wb') as f:
    pickle.dump(artifacts, f)
print(f'Artifacts saved: {art_path}')

Artifacts saved: Artifacts and Submissions/model_artifacts (V38).pkl


In [15]:
print('=' * 60)
print('V38 FINAL SUMMARY')
print('=' * 60)
print(f'  Feature count:     {len(FEAT_COLS)} (42 + 4 new)')
print(f'  New features:      {NEW_FEATURES}')
print(f'  Close-fire seeds:  {CLOSE_SEEDS}')
print(f'  Close alpha/l1:    {CLOSE_ALPHA} / {CLOSE_L1}  (LOCKED)')
print(f'  Best beta:         {best_beta}')
print(f'  OOF hybrid:        {best_h:.4f}')
print(f'  OOF C-index:       {best_c:.4f}')
print(f'  OOF WBS:           {best_wbs:.4f}')
print(f'  Delta vs V28:      {best_h - 0.9734:+.4f}')
print(f'  Violations:        0')
print('=' * 60)

V38 FINAL SUMMARY
  Feature count:     46 (42 + 4 new)
  New features:      ['log_dt', 'perimeter_rate', 'num_x_low_res', 'low_res_x_align']
  Close-fire seeds:  [42, 100, 200, 300, 400]
  Close alpha/l1:    0.2 / 0.5  (LOCKED)
  Best beta:         0.6
  OOF hybrid:        0.9737
  OOF C-index:       0.9451
  OOF WBS:           0.0141
  Delta vs V28:      +0.0003
  Violations:        0
